# PulmoVision: duplicate audit before corrected training
Preserve v1 unchanged. Investigate 1,821 similarity pairs, including 35 cross-source pairs. No model fitting or test evaluation is performed by this notebook. Similarity scores are screening signals, not proof of duplication.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import sys,subprocess,os
BASE=Path('/content/drive/MyDrive/PulmoVision_Journal')
OLD=BASE/'PV_classifier_v1'/'study'
AUDIT=BASE/'PV_duplicate_audit_v2'
AUDIT.mkdir(exist_ok=True)
DATA=Path('/content/pulmovision_data')
CODE=Path('/content/pulmovision_audit_code'); CODE.mkdir(exist_ok=True)


In [ ]:
FILES={'prepare.py': '"""Download NLM datasets, audit duplicates, and save immutable study manifests.\n\nNo patient images leave this machine. This module does not execute input scripts.\n"""\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport re\nimport urllib.request\nimport zipfile\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image, ImageOps\nfrom sklearn.model_selection import StratifiedGroupKFold\n\nSOURCES = {\n    "montgomery": "https://openi.nlm.nih.gov/imgs/collections/NLM-MontgomeryCXRSet.zip",\n    "shenzhen": "https://openi.nlm.nih.gov/imgs/collections/ChinaSet_AllFiles.zip",\n}\nREVISION = "110cbc18d5133582e320b43d53bf5c44e410c936"\n\n\ndef digest(path):\n    h = hashlib.sha256()\n    with open(path, "rb") as f:\n        for chunk in iter(lambda: f.read(1024 * 1024), b""):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef write_json(path, obj):\n    Path(path).parent.mkdir(parents=True, exist_ok=True)\n    Path(path).write_text(json.dumps(obj, indent=2, allow_nan=False), encoding="utf-8")\n\n\ndef write_csv(path, rows, fields=None):\n    rows = list(rows)\n    fields = fields or list(rows[0])\n    with open(path, "w", newline="", encoding="utf-8") as f:\n        writer = csv.DictWriter(f, fieldnames=fields)\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef read_csv(path):\n    with open(path, newline="", encoding="utf-8") as f:\n        return list(csv.DictReader(f))\n\n\ndef download(root):\n    root = Path(root)\n    archive_dir = root / "archives"\n    archive_dir.mkdir(parents=True, exist_ok=True)\n    for source, url in SOURCES.items():\n        archive = archive_dir / f"{source}.zip"\n        if not archive.exists():\n            print(f"Downloading {source} (large archive; Shenzhen about 3.8 GB)", flush=True)\n            partial = archive.with_suffix(".partial")\n            urllib.request.urlretrieve(url, partial)\n            with zipfile.ZipFile(partial) as z:\n                bad = z.testzip()\n                if bad:\n                    raise ValueError(f"Corrupt archive member: {bad}")\n            partial.replace(archive)\n        target = (root / source).resolve()\n        target.mkdir(parents=True, exist_ok=True)\n        marker = target / "extraction.json"\n        archive_hash = digest(archive)\n        if marker.exists():\n            if json.loads(marker.read_text())["archive_sha256"] != archive_hash:\n                raise ValueError("Archive changed; use a new data directory")\n            continue\n        with zipfile.ZipFile(archive) as z:\n            for member in z.infolist():\n                dest = (target / member.filename).resolve()\n                if not dest.is_relative_to(target):\n                    raise ValueError("Unsafe archive path")\n                # Extract only image and lung-mask PNGs; never execute archive content.\n                if member.filename.lower().endswith(".png"):\n                    z.extract(member, target)\n        write_json(marker, {"source_url": url, "archive_sha256": archive_hash})\n        print(f"Extracted {source}", flush=True)\n\n\ndef image_signature(path):\n    with Image.open(path) as im:\n        im = ImageOps.exif_transpose(im).convert("L")\n        size = im.size\n        pixels = np.asarray(im)\n        pixel_hash = hashlib.sha256(str(size).encode() + pixels.tobytes()).hexdigest()\n        thumbnail = np.asarray(im.resize((9, 8), Image.Resampling.LANCZOS))\n        bits = thumbnail[:, 1:] > thumbnail[:, :-1]\n        dhash = int.from_bytes(np.packbits(bits.ravel()).tobytes(), "big")\n    return size, pixel_hash, f"{dhash:016x}"\n\n\ndef inventory(root):\n    root = Path(root).resolve()\n    rows = []\n    for source in SOURCES:\n        for path in sorted((root / source).rglob("*.png")):\n            if path.parent.name != "CXR_png":\n                continue\n            match = re.fullmatch(r"(?:MCUCXR|CHNCXR)_(\\d+)_([01])", path.stem)\n            if not match:\n                raise ValueError(f"Unrecognized label convention: {path.name}")\n            size, pixel_hash, dhash = image_signature(path)\n            rows.append({"image_id": source + ":" + path.name,\n                         "patient_id": source + ":" + match[1],\n                         "source": source, "label": int(match[2]),\n                         "path": path.relative_to(root).as_posix(),\n                         "sha256": digest(path), "pixel_sha256": pixel_hash,\n                         "dhash": dhash, "width": size[0], "height": size[1]})\n    for source in SOURCES:\n        subset = [r for r in rows if r["source"] == source]\n        if not subset or set(r["label"] for r in subset) != {0, 1}:\n            raise ValueError(f"Missing images or class in {source}")\n    return rows\n\n\ndef duplicate_groups(rows, max_distance=4):\n    parent = list(range(len(rows)))\n\n    def find(i):\n        while parent[i] != i:\n            parent[i] = parent[parent[i]]\n            i = parent[i]\n        return i\n\n    candidates = []\n    for i, a in enumerate(rows):\n        for j in range(i):\n            b = rows[j]\n            distance = (int(a["dhash"], 16) ^ int(b["dhash"], 16)).bit_count()\n            exact = a["sha256"] == b["sha256"] or a["pixel_sha256"] == b["pixel_sha256"]\n            same_patient = a["patient_id"] == b["patient_id"]\n            near = distance <= max_distance\n            if exact or same_patient or near:\n                if exact and a["label"] != b["label"]:\n                    raise ValueError(f"Duplicate image with conflicting labels: {a[\'image_id\']}")\n                if exact or same_patient:\n                    parent[find(i)] = find(j)\n                candidates.append({"image_a": a["image_id"], "image_b": b["image_id"],\n                                   "exact": exact, "same_patient": same_patient,\n                                   "dhash_distance": distance,\n                                   "cross_source": a["source"] != b["source"]})\n    members = {}\n    for i, row in enumerate(rows):\n        members.setdefault(find(i), []).append(row["image_id"])\n    for i, row in enumerate(rows):\n        row["group_id"] = hashlib.sha256("|".join(sorted(members[find(i)])).encode()).hexdigest()[:16]\n    return candidates\n\n\ndef make_split(rows, test_source="montgomery", seed=42):\n    # Hold the entire target source out. Conservatively remove source-side members\n    # of suspected cross-source duplicate groups before splitting development data.\n    test_groups = {r["group_id"] for r in rows if r["source"] == test_source}\n    development = [r for r in rows if r["source"] != test_source and r["group_id"] not in test_groups]\n    y = np.array([r["label"] for r in development])\n    groups = np.array([r["group_id"] for r in development])\n    if min(sum(y == 0), sum(y == 1)) < 5:\n        raise ValueError("Insufficient development images for five-fold stratification")\n    splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)\n    train, val = next(splitter.split(np.zeros(len(y)), y, groups))\n    assignment = {development[i]["image_id"]: "train" for i in train}\n    assignment.update({development[i]["image_id"]: "validation" for i in val})\n    result = []\n    for row in rows:\n        partition = "test" if row["source"] == test_source else assignment.get(row["image_id"], "excluded_overlap")\n        result.append({**row, "split": partition})\n    validate_split(result)\n    return result\n\n\ndef validate_split(rows):\n    groups = {}\n    for row in rows:\n        if row["split"] == "excluded_overlap":\n            continue\n        groups.setdefault(row["group_id"], set()).add(row["split"])\n    if any(len(s) > 1 for s in groups.values()):\n        raise ValueError("A patient/duplicate group crosses partitions")\n    for name in ("train", "validation", "test"):\n        if {int(r["label"]) for r in rows if r["split"] == name} != {0, 1}:\n            raise ValueError(f"Partition {name} must have both classes")\n\n\ndef prepare(root, output, test_source="montgomery", seed=42):\n    output = Path(output)\n    if output.exists():\n        raise FileExistsError("Use a new study directory; existing manifests are never overwritten")\n    rows = inventory(root)\n    candidates = duplicate_groups(rows)\n    split = make_split(rows, test_source, seed)\n    output.mkdir(parents=True)\n    write_csv(output / "inventory.csv", rows)\n    write_csv(output / "duplicate_candidates.csv", candidates,\n              ["image_a", "image_b", "exact", "same_patient", "dhash_distance", "cross_source"])\n    write_csv(output / "manifest.csv", split)\n    counts = {s: {str(y): sum(r["split"] == s and r["label"] == y for r in split)\n                  for y in (0, 1)} for s in ("train", "validation", "test", "excluded_overlap")}\n    write_json(output / "protocol.json", {\n        "protocol": "PV-classifier-v2-exploratory", "test_source": test_source, "split_seed": seed,\n        "model": "microsoft/rad-dino", "revision": REVISION,\n        "manifest_sha256": digest(output / "manifest.csv"), "counts": counts,\n        "near_duplicate_policy": "dHash candidates recorded only; exact pixels and same patient grouped. Visual adjudication required before training.",\n        "grouping": "source-qualified filename patient identifier plus exact duplicate components",\n        "limitations": ["No guarantee of detecting all transformed duplicates",\n                        "Patient identity inferred from dataset filenames; verify metadata",\n                        "Cross-source downstream holdout does not prove absence of foundation-model exposure"],\n    })\n    print(json.dumps(counts, indent=2))\n\n\nif __name__ == "__main__":\n    p = argparse.ArgumentParser()\n    p.add_argument("--data", default="data")\n    p.add_argument("--study", default="study")\n    p.add_argument("--download", action="store_true")\n    p.add_argument("--test-source", choices=list(SOURCES), default="montgomery")\n    a = p.parse_args()\n    if a.download:\n        download(a.data)\n    prepare(a.data, a.study, a.test_source)\n', 'audit.py': '"""Audit v1 similarity chains; generate image review sheets, never infer identity from dHash."""\nimport csv\nimport json\nfrom collections import Counter\nfrom pathlib import Path\nimport numpy as np\nfrom PIL import Image, ImageOps, ImageDraw\nfrom prepare import read_csv, write_csv, write_json, digest\n\n\ndef audit_metadata(old, out):\n    old, out = Path(old), Path(out)\n    out.mkdir(parents=True, exist_ok=True)\n    rows = read_csv(old/\'inventory.csv\')\n    pairs = read_csv(old/\'duplicate_candidates.csv\')\n    manifest = read_csv(old/\'manifest.csv\')\n    summary = {\'n_images\': len(rows), \'n_candidate_pairs\': len(pairs),\n        \'exact_pairs\': sum(p[\'exact\']==\'True\' for p in pairs),\n        \'same_patient_pairs\': sum(p[\'same_patient\']==\'True\' for p in pairs),\n        \'cross_source_pairs\': sum(p[\'cross_source\']==\'True\' for p in pairs),\n        \'largest_similarity_components\': Counter(r[\'group_id\'] for r in rows).most_common(10),\n        \'excluded_images\': sum(r[\'split\']==\'excluded_overlap\' for r in manifest),\n        \'source_hashes\': {n:digest(old/n) for n in [\'inventory.csv\',\'manifest.csv\',\'duplicate_candidates.csv\']},\n        \'finding\': \'V1 unioned dHash candidates transitively. Similarity is not evidence of patient identity.\',\n        \'status\': \'Metadata audit complete; visual adjudication remains pending.\'}\n    write_json(out/\'metadata_audit.json\', summary)\n    print(json.dumps(summary, indent=2))\n    return rows, pairs\n\n\ndef visual_audit(old, data, out):\n    out, data = Path(out), Path(data)\n    rows, pairs = audit_metadata(old, out)\n    lookup = {r[\'image_id\']:r for r in rows}\n    cache = {}\n    def get(key):\n        if key not in cache:\n            r=lookup[key]\n            path=data/r[\'path\']\n            if digest(path)!=r[\'sha256\']: raise ValueError(\'Source image changed: \'+key)\n            with Image.open(path) as im:\n                cache[key]=ImageOps.exif_transpose(im).convert(\'L\').resize((256,256))\n        return cache[key]\n    scored=[]\n    for p in pairs:\n        a=np.asarray(get(p[\'image_a\']),dtype=float).ravel()\n        b=np.asarray(get(p[\'image_b\']),dtype=float).ravel()\n        corr=float(np.corrcoef(a,b)[0,1]) if a.std() and b.std() else 0.\n        scored.append({**p,\'normalized_correlation\':corr,\n                       \'priority_review\': p[\'cross_source\']==\'True\' or corr>=.98})\n    write_csv(out/\'all_scored_candidates.csv\',scored)\n    priority=[p for p in scored if p[\'priority_review\']]\n    review=[]\n    sheets=out/\'review_sheets\'; sheets.mkdir(exist_ok=True)\n    for start in range(0,len(priority),4):\n        page=Image.new(\'RGB\',(1100,4*330),\'white\'); draw=ImageDraw.Draw(page)\n        for j,p in enumerate(priority[start:start+4]):\n            idx=start+j\n            y=j*330\n            draw.text((10,y+5),f"Pair {idx:04}: dHash={p[\'dhash_distance\']}, corr={p[\'normalized_correlation\']:.4f}; cross-source={p[\'cross_source\']}",fill=\'black\')\n            for col,k in enumerate([\'image_a\',\'image_b\']):\n                draw.text((col*550+10,y+25),p[k],fill=\'black\')\n                page.paste(get(p[k]).convert(\'RGB\'),(col*550+10,y+50))\n            review.append({\'pair_id\':f\'{idx:04}\', \'image_a\':p[\'image_a\'],\'image_b\':p[\'image_b\'],\n                           \'decision\':\'pending\',\'reviewer\':\'\',\'reason\':\'\',\'sheet\':f\'review_{start//4:03}.jpg\'})\n        page.save(sheets/f\'review_{start//4:03}.jpg\',quality=92)\n    if not (out/\'review_decisions.csv\').exists():\n        write_csv(out/\'review_decisions.csv\',review,[\'pair_id\',\'image_a\',\'image_b\',\'decision\',\'reviewer\',\'reason\',\'sheet\'])\n    write_json(out/\'VISUAL_REVIEW_PENDING.json\',{\'priority_pairs\':len(priority),\'all_candidates\':len(scored),\n        \'policy\':\'All cross-source candidates and within-source correlation>=0.98 prioritized. Scores do not confirm identity. Other candidates retained as unresolved screening evidence.\',\n        \'next\':\'Review sheets and original-resolution images for ambiguous cases. Decisions: duplicate, distinct, uncertain. No corrected training until review is finalized.\'})\n    print(\'Visual review prepared:\',len(priority),\'priority pairs. Training has NOT started.\')\n\n\nif __name__==\'__main__\':\n    import argparse\n    p=argparse.ArgumentParser(); p.add_argument(\'--old\',required=True); p.add_argument(\'--out\',required=True); p.add_argument(\'--data\')\n    a=p.parse_args()\n    if a.data: visual_audit(a.old,a.data,a.out)\n    else: audit_metadata(a.old,a.out)\n', 'experiment.py': '"""Frozen RAD-DINO features, two classifier heads, locked external evaluation.\n\nCommands: encode, fit, evaluate. No image-based clinical claims are made by this code.\n"""\nimport argparse\nimport importlib.metadata\nimport json\nimport os\nimport platform\nimport random\nfrom pathlib import Path\n\nimport numpy as np\nfrom prepare import digest, read_csv, write_csv, write_json, validate_split, REVISION\n\n\ndef metadata():\n    result = {"python": platform.python_version(), "platform": platform.platform()}\n    for package in ("torch", "transformers", "numpy", "scikit-learn", "pillow", "huggingface-hub"):\n        try:\n            result[package] = importlib.metadata.version(package)\n        except importlib.metadata.PackageNotFoundError:\n            result[package] = "not installed"\n    result["experiment_code_sha256"] = digest(__file__)\n    result["prepare_code_sha256"] = digest(Path(__file__).with_name("prepare.py"))\n    return result\n\n\ndef locked_manifest(study):\n    study = Path(study)\n    protocol = json.loads((study / "protocol.json").read_text())\n    if digest(study / "manifest.csv") != protocol["manifest_sha256"]:\n        raise ValueError("Study manifest changed after protocol was created")\n    rows = read_csv(study / "manifest.csv")\n    validate_split(rows)\n    return rows, protocol\n\n\ndef seed_everything(seed):\n    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")\n    import torch\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    torch.use_deterministic_algorithms(True)\n    torch.backends.cudnn.benchmark = False\n    torch.set_num_threads(min(4, os.cpu_count() or 1))\n\n\ndef head_model(dim):\n    import torch.nn as nn\n    return nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, 256), nn.GELU(),\n                         nn.Dropout(0.3), nn.Linear(256, 2))\n\n\ndef encode(data, study, batch_size=4, allow_cpu=False):\n    import torch\n    from PIL import Image, ImageOps\n    from transformers import AutoImageProcessor, AutoModel\n    rows, protocol = locked_manifest(study)\n    if not torch.cuda.is_available() and not allow_cpu:\n        raise RuntimeError("GPU required for full feature extraction; use Colab GPU or explicitly --allow-cpu")\n    seed_everything(42)\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    root = Path(study) / "features"\n    root.mkdir(exist_ok=True)\n    cache_meta = {"manifest_sha256": protocol["manifest_sha256"],\n                  "model": protocol["model"], "revision": REVISION,\n                  "preprocessing": "AutoImageProcessor(use_fast=False), RGB, EXIF transpose, no augmentation",\n                  "device": device, "environment": metadata()}\n    old_meta = root / "metadata.json"\n    if old_meta.exists():\n        old = json.loads(old_meta.read_text())\n        if any(old[k] != cache_meta[k] for k in ("manifest_sha256", "model", "revision", "preprocessing", "environment", "device")):\n            raise ValueError("Feature cache configuration changed. Use a new study directory")\n    write_json(old_meta, cache_meta)\n    processor = AutoImageProcessor.from_pretrained(protocol["model"], revision=REVISION, use_fast=False)\n    model = AutoModel.from_pretrained(protocol["model"], revision=REVISION, use_safetensors=True)\n    model.eval().to(device)\n    for p in model.parameters():\n        p.requires_grad_(False)\n    processor.save_pretrained(root / "processor")\n    pending = []\n    for row in rows:\n        if row["split"] == "excluded_overlap":\n            continue\n        path = Path(data) / row["path"]\n        if digest(path) != row["sha256"]:\n            raise ValueError(f"Image content changed: {row[\'image_id\']}")\n        dest = root / f"{row[\'sha256\']}.npy"\n        if not dest.exists():\n            pending.append(row)\n    with torch.inference_mode():\n        for i in range(0, len(pending), batch_size):\n            batch = pending[i:i + batch_size]\n            images = []\n            for row in batch:\n                with Image.open(Path(data) / row["path"]) as im:\n                    images.append(ImageOps.exif_transpose(im).convert("RGB"))\n            inputs = processor(images=images, return_tensors="pt").to(device)\n            vectors = model(**inputs).last_hidden_state[:, 0, :].float().cpu().numpy()\n            for row, vector in zip(batch, vectors):\n                dest = root / f"{row[\'sha256\']}.npy"\n                partial = dest.with_suffix(".partial")\n                with open(partial, "wb") as f:\n                    np.save(f, vector, allow_pickle=False)\n                partial.replace(dest)\n            print(f"Encoded {min(i + len(batch), len(pending))}/{len(pending)} pending images", flush=True)\n    print("Feature extraction complete", flush=True)\n\n\ndef load_features(study, rows, split):\n    selected = [r for r in rows if r["split"] == split]\n    x = np.stack([np.load(Path(study) / "features" / f"{r[\'sha256\']}.npy", allow_pickle=False) for r in selected])\n    y = np.array([int(r["label"]) for r in selected])\n    if x.ndim != 2 or not np.isfinite(x).all():\n        raise ValueError("Invalid feature cache")\n    return x.astype(np.float32), y, selected\n\n\ndef select_threshold(y, p, target=0.90):\n    from sklearn.metrics import roc_curve\n    fpr, tpr, thresholds = roc_curve(y, p, drop_intermediate=False)\n    eligible = np.where((tpr >= target) & np.isfinite(thresholds))[0]\n    if len(eligible) == 0:\n        raise ValueError("No finite threshold reaches target validation sensitivity")\n    # Lowest false-positive rate, then highest threshold among ties.\n    idx = min(eligible, key=lambda k: (fpr[k], -thresholds[k]))\n    return float(thresholds[idx])\n\n\ndef metric_values(y, p, threshold):\n    from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, confusion_matrix\n    tn, fp, fn, tp = confusion_matrix(y, p >= threshold, labels=[0, 1]).ravel()\n    return {"auroc": float(roc_auc_score(y, p)),\n            "average_precision": float(average_precision_score(y, p)),\n            "sensitivity": float(tp / (tp + fn)), "specificity": float(tn / (tn + fp)),\n            "ppv": float(tp / (tp + fp)) if tp + fp else None,\n            "npv": float(tn / (tn + fn)) if tn + fn else None,\n            "brier": float(brier_score_loss(y, p)),\n            "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn)}\n\n\ndef bootstrap(y, p, groups, threshold, repeats=2000, seed=2026):\n    rng = np.random.default_rng(seed)\n    unique = np.unique(groups)\n    indices = {g: np.where(groups == g)[0] for g in unique}\n    samples = {k: [] for k in ("auroc", "average_precision", "sensitivity", "specificity", "brier")}\n    for _ in range(repeats):\n        ix = np.concatenate([indices[g] for g in rng.choice(unique, len(unique), replace=True)])\n        if len(np.unique(y[ix])) < 2:\n            continue\n        vals = metric_values(y[ix], p[ix], threshold)\n        for k in samples:\n            samples[k].append(vals[k])\n    if len(samples["auroc"]) < 0.9 * repeats:\n        raise ValueError("Too few valid bootstrap samples; dataset is unsuitable for this interval method")\n    return {k: [float(v) for v in np.percentile(values, [2.5, 97.5])] for k, values in samples.items()}\n\n\ndef feature_fingerprint(study, rows):\n    import hashlib\n    h = hashlib.sha256((Path(study) / "features" / "metadata.json").read_bytes())\n    for row in rows:\n        if row["split"] != "excluded_overlap":\n            path = Path(study) / "features" / f"{row[\'sha256\']}.npy"\n            h.update((row["image_id"] + digest(path)).encode())\n    return h.hexdigest()\n\n\ndef fit(study, seeds=(42, 43, 44), epochs=20):\n    import torch\n    from sklearn.metrics import roc_auc_score\n    from sklearn.linear_model import LogisticRegression\n    from sklearn.preprocessing import StandardScaler\n    rows, protocol = locked_manifest(study)\n    out = Path(study) / "models"\n    if out.exists():\n        raise FileExistsError("Models already exist. Do not overwrite a locked study")\n    xtrain, ytrain, _ = load_features(study, rows, "train")\n    xval, yval, _ = load_features(study, rows, "validation")\n    cache_hash = feature_fingerprint(study, rows)\n    out.mkdir()\n    config = {"manifest_sha256": protocol["manifest_sha256"], "feature_fingerprint": cache_hash,\n              "environment": metadata(), "seeds": list(seeds), "epochs": epochs,\n              "optimizer": "AdamW", "lr": 1e-4, "weight_decay": 1e-4, "batch_size": 8,\n              "selection": "highest validation AUROC; earliest epoch for ties",\n              "threshold": "validation sensitivity >=0.90, maximize specificity",\n              "augmentation": "none; cached deterministic frozen RAD-DINO features",\n              "test_access_during_fit": "feature file integrity checks only; no test metrics or model selection"}\n    write_json(out / "training_config.json", config)\n    scaler = StandardScaler().fit(xtrain)\n    linear = LogisticRegression(C=1.0, class_weight="balanced", max_iter=2000, random_state=42)\n    linear.fit(scaler.transform(xtrain), ytrain)\n    if max(linear.n_iter_) >= 2000:\n        raise RuntimeError("Logistic regression did not converge")\n    lp = linear.predict_proba(scaler.transform(xval))[:, 1]\n    np.savez(out / "linear.npz", mean=scaler.mean_, scale=scaler.scale_,\n             coef=linear.coef_, intercept=linear.intercept_)\n    write_json(out / "linear.json", {"threshold": select_threshold(yval, lp),\n                                    "validation_auroc": float(roc_auc_score(yval, lp)), "C": 1.0})\n    for seed in seeds:\n        seed_everything(seed)\n        model = head_model(xtrain.shape[1])\n        train = torch.utils.data.TensorDataset(torch.from_numpy(xtrain), torch.from_numpy(ytrain))\n        loader = torch.utils.data.DataLoader(train, batch_size=8, shuffle=True,\n                                             generator=torch.Generator().manual_seed(seed))\n        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)\n        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)\n        weights = 1.0 / np.bincount(ytrain, minlength=2)\n        weights = torch.tensor(weights / weights.sum(), dtype=torch.float32)\n        criterion = torch.nn.CrossEntropyLoss(weight=weights)\n        best_auc, best_epoch = -1.0, None\n        log = []\n        for epoch in range(1, epochs + 1):\n            model.train()\n            total = 0.0\n            for x, y in loader:\n                optimizer.zero_grad(set_to_none=True)\n                loss = criterion(model(x), y)\n                loss.backward()\n                optimizer.step()\n                total += loss.item() * len(y)\n            scheduler.step()\n            model.eval()\n            with torch.inference_mode():\n                probabilities = model(torch.from_numpy(xval)).softmax(-1)[:, 1].numpy()\n            auc = float(roc_auc_score(yval, probabilities))\n            log.append({"epoch": epoch, "train_loss": total / len(ytrain), "validation_auroc": auc})\n            if auc > best_auc:\n                best_auc, best_epoch = auc, epoch\n                torch.save(model.state_dict(), out / f"head_seed{seed}.pt")\n            print(f"Seed {seed}, epoch {epoch}: validation AUROC {auc:.4f}", flush=True)\n        model.load_state_dict(torch.load(out / f"head_seed{seed}.pt", weights_only=True), strict=True)\n        model.eval()\n        with torch.inference_mode():\n            p = model(torch.from_numpy(xval)).softmax(-1)[:, 1].numpy()\n        write_csv(out / f"training_seed{seed}.csv", log)\n        write_json(out / f"head_seed{seed}.json", {"threshold": select_threshold(yval, p),\n                   "selected_epoch": best_epoch, "validation_auroc": best_auc, "input_dim": xtrain.shape[1]})\n    write_json(out / "FIT_COMPLETE.json", {"files": {p.name: digest(p) for p in sorted(out.iterdir()) if p.is_file()}})\n\n\ndef evaluate(study, repeats=2000):\n    import torch\n    rows, protocol = locked_manifest(study)\n    models = Path(study) / "models"\n    done = json.loads((models / "FIT_COMPLETE.json").read_text())\n    for name, expected in done["files"].items():\n        if digest(models / name) != expected:\n            raise ValueError(f"Model artifact changed: {name}")\n    config = json.loads((models / "training_config.json").read_text())\n    if config["manifest_sha256"] != protocol["manifest_sha256"] or config["feature_fingerprint"] != feature_fingerprint(study, rows):\n        raise ValueError("Data or cached features changed after fitting")\n    results = Path(study) / "results"\n    if results.exists():\n        raise FileExistsError("Test evaluation already exists; preserve the original results")\n    x, y, selected = load_features(study, rows, "test")\n    groups = np.array([r["group_id"] for r in selected])\n    results.mkdir()\n    names = ["linear"] + [f"head_seed{s}" for s in config["seeds"]]\n    summary = []\n    all_probabilities = {}\n    for name in names:\n        meta = json.loads((models / f"{name}.json").read_text())\n        if name == "linear":\n            with np.load(models / "linear.npz", allow_pickle=False) as z:\n                logits = (((x - z["mean"]) / z["scale"]) @ z["coef"].T + z["intercept"]).ravel()\n            p = 1 / (1 + np.exp(-np.clip(logits, -700, 700)))\n        else:\n            model = head_model(meta["input_dim"])\n            model.load_state_dict(torch.load(models / f"{name}.pt", weights_only=True), strict=True)\n            model.eval()\n            with torch.inference_mode():\n                p = model(torch.from_numpy(x)).softmax(-1)[:, 1].numpy()\n        all_probabilities[name] = p\n        threshold = meta["threshold"]\n        vals = metric_values(y, p, threshold)\n        intervals = bootstrap(y, p, groups, threshold, repeats)\n        record = {"model": name, "threshold": threshold, "n_images": len(y), "n_groups": len(set(groups)),\n                  "metrics": vals, "cluster_bootstrap_95ci": intervals,\n                  "threshold_0_5_metrics": metric_values(y, p, 0.5)}\n        write_json(results / f"{name}_metrics.json", record)\n        write_csv(results / f"{name}_predictions.csv", [{"image_id": r["image_id"],\n                  "source": r["source"], "group_id": r["group_id"], "label": int(r["label"]),\n                  "tb_probability": float(prob), "threshold": threshold,\n                  "prediction": int(prob >= threshold)} for r, prob in zip(selected, p)])\n        summary.append({"model": name, "n": len(y), "threshold": threshold, **vals,\n                        **{f"{k}_ci_low": v[0] for k, v in intervals.items()},\n                        **{f"{k}_ci_high": v[1] for k, v in intervals.items()}})\n    write_csv(results / "summary.csv", summary)\n    write_json(results / "evaluation_provenance.json", {"environment": metadata(),\n               "manifest_sha256": protocol["manifest_sha256"], "bootstrap_repeats": repeats,\n               "bootstrap_seed": 2026, "unit": "image metrics, patient/duplicate-group bootstrap",\n               "warning": "Confidence intervals condition on fitted models and validation-selected thresholds. No clinical validation."})\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n    from sklearn.metrics import roc_curve\n    fig, ax = plt.subplots(figsize=(6, 5))\n    for name, probabilities in all_probabilities.items():\n        fpr, tpr, _ = roc_curve(y, probabilities)\n        ax.plot(fpr, tpr, label=name)\n    ax.plot([0, 1], [0, 1], "k--", alpha=0.4)\n    ax.set(xlabel="False-positive rate", ylabel="Sensitivity", title="Held-out source: " + protocol["test_source"])\n    ax.legend()\n    fig.tight_layout()\n    fig.savefig(results / "roc.png", dpi=200)\n    plt.close(fig)\n    print(f"Saved predictions, intervals and plots to {results}")\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("command", choices=["encode", "fit", "evaluate"])\n    parser.add_argument("--study", default="study")\n    parser.add_argument("--data", default="data")\n    parser.add_argument("--batch-size", type=int, default=4)\n    parser.add_argument("--allow-cpu", action="store_true")\n    args = parser.parse_args()\n    if args.command == "encode":\n        encode(args.data, args.study, args.batch_size, args.allow_cpu)\n    elif args.command == "fit":\n        fit(args.study)\n    else:\n        evaluate(args.study)\n', 'test_pipeline.py': '"""Synthetic software checks only: these are NOT medical experiment results."""\nimport copy\nimport json\nimport tempfile\nimport unittest\nfrom pathlib import Path\nimport numpy as np\nimport prepare as prep\nimport experiment as exp\n\n\nclass PipelineChecks(unittest.TestCase):\n    def rows(self):\n        rng = np.random.default_rng(123)\n        rows = []\n        for source, n in [(\'shenzhen\', 60), (\'montgomery\', 20)]:\n            for i in range(n):\n                key = f\'{source}:{i}\'\n                rows.append(dict(image_id=key, patient_id=key, source=source,\n                    label=i % 2, path=key, sha256=key.replace(\':\', \'_\'),\n                    pixel_sha256=key, dhash=rng.bytes(8).hex()))\n        return rows\n\n    def test_grouped_holdout(self):\n        rows = self.rows()\n        # Simulated duplicate between development and test must be excluded.\n        rows[0].update({k: rows[-2][k] for k in [\'sha256\', \'pixel_sha256\', \'dhash\']})\n        prep.duplicate_groups(rows)\n        result = prep.make_split(rows)\n        self.assertEqual(result[0][\'split\'], \'excluded_overlap\')\n        self.assertTrue(all(r[\'split\'] == \'test\' for r in result if r[\'source\'] == \'montgomery\'))\n        self.assertEqual(result, prep.make_split(rows))\n        broken = copy.deepcopy(result)\n        train = next(r for r in broken if r[\'split\'] == \'train\')\n        val = next(r for r in broken if r[\'split\'] == \'validation\')\n        val[\'group_id\'] = train[\'group_id\']\n        with self.assertRaises(ValueError): prep.validate_split(broken)\n\n    def test_conflicting_duplicate_labels(self):\n        rows = self.rows()\n        rows[1][\'pixel_sha256\'] = rows[0][\'pixel_sha256\']\n        with self.assertRaises(ValueError): prep.duplicate_groups(rows)\n\n    def test_known_metrics(self):\n        y = np.array([0, 0, 1, 1])\n        p = np.array([.1, .4, .35, .8])\n        m = exp.metric_values(y, p, .5)\n        self.assertAlmostEqual(m[\'auroc\'], .75)\n        self.assertEqual(m[\'sensitivity\'], .5)\n        self.assertEqual(m[\'specificity\'], 1.)\n        self.assertAlmostEqual(exp.select_threshold(y, p), .35)\n\n    def test_training_evaluation_and_lock(self):\n        with tempfile.TemporaryDirectory() as d:\n            study = Path(d)\n            rows = self.rows()\n            prep.duplicate_groups(rows)\n            rows = prep.make_split(rows)\n            prep.write_csv(study/\'manifest.csv\', rows)\n            prep.write_json(study/\'protocol.json\', {\'manifest_sha256\': prep.digest(study/\'manifest.csv\'),\n                \'test_source\': \'montgomery\'})\n            features = study/\'features\'\n            features.mkdir()\n            prep.write_json(features/\'metadata.json\', {\'synthetic_qa_only\': True})\n            rng = np.random.default_rng(456)\n            for row in rows:\n                x = rng.normal(size=16).astype(np.float32)\n                x[0] += int(row[\'label\']) * 2\n                np.save(features/f"{row[\'sha256\']}.npy", x, allow_pickle=False)\n            exp.fit(study, seeds=(42,), epochs=2)\n            exp.evaluate(study, repeats=100)\n            self.assertTrue((study/\'results\'/\'summary.csv\').exists())\n            with self.assertRaises(FileExistsError): exp.fit(study, seeds=(42,), epochs=2)\n            with self.assertRaises(FileExistsError): exp.evaluate(study, repeats=100)\n            with open(study/\'manifest.csv\', \'a\') as f: f.write(\'\\n\')\n            with self.assertRaises(ValueError): exp.locked_manifest(study)\n\n\nif __name__ == \'__main__\':\n    unittest.main()\n'}
for name,content in FILES.items():
 (CODE/name).write_text(content)
os.chdir(CODE)
subprocess.run([sys.executable,"-m","pip","install","-q","scikit-learn==1.7.2"],check=True)
subprocess.run([sys.executable,"audit.py","--old",str(OLD),"--out",str(AUDIT)],check=True)


## Download and inspect source images
Source images remain on the Colab runtime; the audit records and review sheets are saved to Drive. This can take around 45 minutes at the previously observed transfer speed. The original classifier run will not be overwritten.

In [ ]:
import prepare
prepare.download(DATA)
subprocess.run([sys.executable,"audit.py","--old",str(OLD),"--out",str(AUDIT),"--data",str(DATA)],check=True)


## Review sheets
Review all cross-source candidates and very high-correlation within-source candidates. Other candidates remain documented; do not claim that this screen detects all duplicates. Open original-resolution images if a thumbnail comparison is ambiguous. The next stage will record adjudication before creating a corrected manifest. Montgomery is now an exploratory benchmark, not an untouched confirmatory test.

In [ ]:
from IPython.display import display,Image
import json
print((AUDIT/"VISUAL_REVIEW_PENDING.json").read_text())
for sheet in sorted((AUDIT/"review_sheets").glob("*.jpg")):
 print(sheet.name)
 display(Image(filename=str(sheet)))
print("Audit sheets saved. Corrected training awaits visual review.")
